# 06 — Reasoning-Oriented Prompting

## Scenario
A service is throwing 500 errors. We have the recent system logs. We need an AI to recommend a triage action. 

**The Danger:** Naive prompts often cause models to jump to conclusions (like "Restart the server") without checking the evidence. We need to force the model to compute observable reasoning steps before it answers.

In [ ]:
import os
import json
from typing import List
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

SYSTEM_LOGS = """
[10:01:05] INFO: Service started.
[10:15:22] WARN: Database connection latency spike (200ms).
[10:15:25] ERROR: Connection pool exhausted. Failed to acquire connection to DB_MAIN.
[10:15:26] ERROR: HTTP 500 returned to client (Timeout).
"""

class TriageRecommendation(BaseModel):
    recommended_action: str = Field(description="The exact action the on-call engineer should take.")
    confidence: str = Field(description="High, Medium, or Low.")


## Step 1: The Naive Direct Prompt

We ask for the answer directly. Models trained on internet text often default to "turn it off and on again" for technical issues if they aren't forced to read the context carefully.

In [ ]:
naive_prompt = f"""You are an SRE. The service is throwing 500 errors.\nLogs:\n{SYSTEM_LOGS}\n\nWhat should we do?"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=naive_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=TriageRecommendation,
    )
)

print("Naive Output:", response.text)
# Notice: It might recommend restarting the app service, but the logs clearly show a DB connection pool exhaustion. Restarting the app might just flood the DB with new connection requests, worsening the issue.

## Step 2: Structured Chain-of-Thought (CoT)

We can force the model to reason *before* it outputs the `recommended_action`. 
**State-of-the-Art Technique:** When using JSON/Structured outputs, we add a `reasoning_steps` array to the top of our Pydantic schema. Because LLMs generate tokens sequentially, forcing the JSON to output the reasoning array *first* guarantees the model computes its logic before committing to the final answer.

In [ ]:
class StructuredCoTTriage(BaseModel):
    # Force reasoning to happen first!
    reasoning_steps: List[str] = Field(description="Step-by-step analysis of the logs to determine the root cause.")
    recommended_action: str = Field(description="The exact action the on-call engineer should take, based strictly on the reasoning.")
    confidence: str = Field(description="High, Medium, or Low.")

cot_prompt = f"""You are an SRE. The service is throwing 500 errors.\nLogs:\n{SYSTEM_LOGS}\n\nAnalyze the logs step-by-step to find the root cause, then recommend an action."""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=cot_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=StructuredCoTTriage,
    )
)

output = StructuredCoTTriage.model_validate_json(response.text)
print("\n--- Observable Reasoning Traces ---")
for i, step in enumerate(output.reasoning_steps):
    print(f"{i+1}. {step}")
print(f"\n--- Final Recommendation ({output.confidence}) ---")
print(output.recommended_action)

# Notice: By forcing the model to explicitly state the connection pool error in its reasoning, its final recommendation shifts to investigating the DB or increasing the pool size, which is the correct action.

## Step 3: Compound System (Planner / Verifier)

For critical actions (like actually executing code), a single LLM call is too risky. We split the reasoning into a Compound AI System: one call plans/proposes, a separate call (with a different prompt) verifies the proposal against safety rules.

In [ ]:
# The proposal is the output from Step 2.
proposed_action = output.recommended_action

class VerificationResult(BaseModel):
    is_safe: bool = Field(description="True if the action is safe to execute automatically. False if it risks data loss or cascading failure.")
    reasoning: str

verifier_prompt = f"""You are the Safety Verifier.\nProposed Action: {proposed_action}\n\nRules:\n- Scaling up a database or increasing pool sizes automatically without human approval is DANGEROUS.\n- Restarting read-only services is SAFE.\n- Read-only queries to investigate are SAFE.\n\nEvaluate the proposed action."""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=verifier_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=VerificationResult,
    )
)

verification = VerificationResult.model_validate_json(response.text)
print("\n--- Verifier Decision ---")
print(f"Safe to auto-execute? {verification.is_safe}")
print(f"Reasoning: {verification.reasoning}")

# If is_safe is False, the application pauses and pages a human with the artifacts.
# This separates the 'smart planning' from the 'strict safety enforcement'.